# Quickstart: Latent Domain Recovery

This notebook demonstrates the core idea in a minimal end-to-end example.
We train a **LiftingLayer** on 1-D Gaussian waveforms and inspect the group representations, and equivalent lifting matrix it learns.

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from core.train_utils import create_model, make_data_generator, train

print('TensorFlow:', tf.__version__)
print('GPUs available:', tf.config.list_physical_devices('GPU'))

## 2. Define a minimal experiment

We use 1-D Gaussian waveforms observed in a randomly permuted (shuffled pixel order) domain.  
The model must discover that the underlying structure is translation on a 1-D grid.

In [ ]:
exp_specs = {
    "seed": 42,
    "training_duration_in_epochs": 5000,
    "num_training_batches": 100,
    "model_optimizer_starting_lr": 2.5e-4,
    "model_optimizer_ending_lr": 2.5e-5,
    "estimators_optimizer_starting_lr": 2.5e-3,
    "estimators_optimizer_ending_lr": 2.5e-4,
    "model_global_gradclip": 0.5,
    "estimator_global_gradclip": 0.5,
    "data_generator_params": {
        "dims": 1,
        "output_representation": "natural",
        "n_components": 33,
        "batch_size": 1000,
        "noise_normalized_std": 0.05,
        "features": [
            {"type": "gaussian", "scale_min": 0.5, "scale_max": 2.5,
             "amplitude_min": 0.5, "amplitude_max": 1.5}
        ]
    },
    "model_params": {
        "covariance_shift": 1.0e-5,
        "tcr_noise_var": 1.0e-5,
        "num_lenses": 1,
        "dilations": [1],
        "normalization_noise": [None],
        "lifting_space_dims": 63,
        "lifting_layer_params": {
            "axes_widths": [33],
            "use_spectral_lifting": True,
            "lifting_offset_limit": 0.5
        },
        "uniformity_estimator_params": [{
            "group_dims": 1,
            "layer_properties": [
                {"filters": 4, "kernel_size": 5, "strides": 2, "pooling": False},
                {"filters": 4, "kernel_size": 5, "strides": 2, "pooling": False},
                {"filters": 6, "kernel_size": 3, "strides": 1, "pooling": True},
                {"filters": 8, "kernel_size": 3, "strides": 1, "pooling": True}
            ]
        }],
        "probability_estimator_params": [{"num_kernels": 4}]
    },
    "model_loss_coeffs": {
        "uniformity_maximization_reg_coeff": 1.0,
        "total_correlation_minimization_reg_coeff": 0.5,
        "joint_entropy_maximization_reg_coeff": 0.5
    }
}

## 3. Generate training data

In [ ]:
seed = exp_specs["seed"]
np.random.seed(seed)
tf.random.set_seed(seed)

dg = make_data_generator(seed=seed, **exp_specs["data_generator_params"])
dg.reset_batch_counter()
batches = [dg.sample_batch_of_data() for _ in range(exp_specs["num_training_batches"])]
dataset_np = np.concatenate(batches, axis=0)
print(f"Dataset shape: {dataset_np.shape}  (samples x input_dim)")

# Preview a few samples
fig, ax = plt.subplots(figsize=(8, 3))
for i in range(6):
    ax.plot(dataset_np[i], alpha=0.7, label=f'sample {i}')
ax.set_title('Training samples (Gaussian waveforms, natural domain)')
ax.set_xlabel('Component index'); ax.set_ylabel('Amplitude')
plt.tight_layout(); plt.show()

## 4. Train the model

In [ ]:
import os
saving_dir = os.path.join('..', 'experiments', 'quickstart', 'epochs')
os.makedirs(saving_dir, exist_ok=True)

model = create_model(**exp_specs["model_params"])

train(
    model=model,
    dataset_np=dataset_np,
    exp_specs=exp_specs,
    saving_dir=saving_dir,
    eager=False  # eager=True avoids tf.function trace conflicts across cells
)
print('Training complete.')

## 5. Inspect the learned group structure

After training the lifting layer encodes a **generator matrix** — the infinitesimal generator of the learned symmetry group.  
For 1-D translation it should converge to a circulant shift matrix.

In [ ]:
G = model.lifting_layer.generators.numpy()   # shape: (group_dims, D, D)
log_G = model.lifting_layer.log_generators.numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(G[0], cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_title('Learned generator $G$')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(log_G[0], cmap='RdBu_r')
axes[1].set_title('Log-generator $\\log G$')
plt.colorbar(im1, ax=axes[1])

plt.suptitle('Learned group structure (should resemble a circulant shift)')
plt.tight_layout(); plt.show()

In [ ]:
# Also inspect the learned lifting map L: input_dim -> (axes_width x channels)
sample = tf.constant(dataset_np[:64], dtype=tf.float32)
lifted, L = model(sample, training=False, analyze=True)

print(f'Lifting map shape: {L.shape}  (input_dim, group_positions)')
print(f'Lifted representation shape: {lifted.shape}  (batch, positions, channels)')

fig, ax = plt.subplots(figsize=(8, 3))
ax.imshow(L.numpy().T, aspect='auto', cmap='RdBu_r')
ax.set_xlabel('Input dimension'); ax.set_ylabel('Group position')
ax.set_title('Learned lifting map $L$ (rows = group positions, cols = input dims)')
plt.tight_layout(); plt.show()